In [ ]:
#python3 -m venv env
#source env/bin/activate
#pip install taxopy
#deactivate
#jupyter notebook --no-browser --port=8888 --ip=0.0.0.0


In [1]:
##!pip install taxopy[fuzzy-matching]
import os

import taxopy

import pandas as pd

##!pip install pivottablejs
#from pivottablejs import pivot_ui

from rapidfuzz import process

import seaborn as sns

In [2]:
#wget ftp://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz
taxdb = taxopy.TaxDb(nodes_dmp="/home/dcm/250513Pat_250728Pat_250930Pat/taxdump/nodes.dmp", names_dmp="/home/dcm/250513Pat_250728Pat_250930Pat/taxdump/names.dmp")

In [7]:
os.makedirs('nt_prok_blastn_out', exist_ok=True)

final_df = pd.DataFrame()

project_list = [
    '/home/dcm/250513Pat',
    '/home/dcm/250728Pat',
    '/home/dcm/250930Pat'
    ]

for project in project_list:

    file_df = pd.read_csv(f'{project}/files.txt', sep='\t')
    

    
    for folder, file in zip(file_df['folder'], file_df['file']):
        col_names = [
            "#ID", 
            "Avg_fold"
        ]
        
        df_cov = pd.read_csv(f"{project}/bbmap_coverage_out_bbnorm_100x/{file}_constats.txt", sep="\t", usecols=col_names, dtype=str, converters={"Avg_fold": float} )	
        
        col_names = [
            "qseqid", "sseqid", "salltitles", "staxids", "pident", "length",
            "qlen", "slen", "mismatch", "gapopen", "qstart", "qend",
            "sstart", "send", "evalue", "bitscore"
        ]
        
        df = pd.read_csv(f"{project}/nt_prok_blastn_out/{file}_scaffolds_1000bp_nt_prok_blastn_out.txt", sep="\t", names=col_names, header=None, dtype=str, converters={"pident": float} )
        
        df = pd.merge(df, df_cov, left_on = "qseqid", right_on = "#ID", how = 'left') 
        
        df['salltitles_$_staxids'] = df['salltitles'] + '_$_' + df['staxids']
        
        df['length'] = df['qseqid'].str.extract(r'length_(\d+)_cov').astype(int)
        
        #filter
        df = df[df['pident'] >= 95]
        df = df[df['length'] >= 1000]
        df = df[df['Avg_fold'] >= 10]
        
        most_common_taxa = (
        df.groupby('qseqid')['salltitles_$_staxids']
        .agg(lambda x: x.mode().iloc[0])  # most common taxa per qseqid
        .reset_index(name='taxid')  # convert Series to DataFrame with column name
        )
        most_common_taxa['file'] = f'{file}'
        
        final_df = pd.concat([final_df, most_common_taxa], ignore_index=True)
    
final_df.to_csv("nt_prok_blastn_out/nt_prok_blastn_out.txt", sep='\t', index = False)

/tmp/ipykernel_3948948/526360487.py:23: ParserWarning: Both a converter and dtype were specified for column Avg_fold - only the converter will be used.
  df_cov = pd.read_csv(f"{project}/bbmap_coverage_out_bbnorm_100x/{file}_constats.txt", sep="\t", usecols=col_names, dtype=str, converters={"Avg_fold": float} )
/tmp/ipykernel_3948948/526360487.py:31: ParserWarning: Both a converter and dtype were specified for column pident - only the converter will be used.
  df = pd.read_csv(f"{project}/nt_prok_blastn_out/{file}_scaffolds_1000bp_nt_prok_blastn_out.txt", sep="\t", names=col_names, header=None, dtype=str, converters={"pident": float} )
/tmp/ipykernel_3948948/526360487.py:23: ParserWarning: Both a converter and dtype were specified for column Avg_fold - only the converter will be used.
  df_cov = pd.read_csv(f"{project}/bbmap_coverage_out_bbnorm_100x/{file}_constats.txt", sep="\t", usecols=col_names, dtype=str, converters={"Avg_fold": float} )
/tmp/ipykernel_3948948/526360487.py:31: Par

In [10]:
del final_df

In [11]:
df_tax = pd.read_csv('nt_prok_blastn_out/nt_prok_blastn_out.txt', sep='\t')
df_tax

,qseqid,taxid,file
0,NODE_1001_length_5588_cov_8.509489,"Rothia kristinae strain ND-2018N chromosome, c...",250513Pat_D25-7849
1,NODE_1004_length_5581_cov_13.102787,Peptostreptococcus anaerobius strain SB204 chr...,250513Pat_D25-7849
2,NODE_1005_length_5575_cov_7.503623,Rothia kristinae strain FDAARGOS_864 chromosom...,250513Pat_D25-7849
3,NODE_1006_length_5564_cov_7.022690,"Cutibacterium acnes SZ2 DNA, complete genome_$...",250513Pat_D25-7849
4,NODE_1007_length_5536_cov_23.626893,Veillonella rogosae strain AC2811 AN NA 2 chro...,250513Pat_D25-7849
...,...,...,...
318606,NODE_997_length_5892_cov_14.912798,Pseudomonas juntendi strain 18091276 chromosom...,250930Pat_D25-12478
318607,NODE_998_length_5889_cov_7.993315,Raoultella terrigena strain Res13-Abat-PEB01-P...,250930Pat_D25-12478
318608,NODE_999_length_5885_cov_15.724357,Veillonella dispar strain NCTC11831 genome ass...,250930Pat_D25-12478
318609,NODE_99_length_89346_cov_114.083771,Klebsiella michiganensis strain RHBSTW-00676 c...,250930Pat_D25-12478


In [12]:
df_tax[['taxname', 'taxid']] = df_tax['taxid'].str.split('_\$_', expand=True)

In [13]:
df_tax = df_tax[['taxid']]
df_tax.drop_duplicates(inplace=True)
df_tax

/tmp/ipykernel_3948948/745975608.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tax.drop_duplicates(inplace=True)


,taxid
0,37923
1,1261
3,1747
4,423477
6,40324
...,...
317739,2774835
317747,2479393
318240,2774993
318542,74829


In [14]:
#check if any empty taxids
df_tax[df_tax['taxid'].isna()]

,taxid


In [15]:
#check if taxid have commma list
mask = df_tax['taxid'].str.contains(',', na=False) 
df_tax[mask]

,taxid


In [16]:
#check if taxid have semi-colon list
mask = df_tax['taxid'].str.contains(';', na=False) 
df_tax[mask]

,taxid
1605,77133;135075
21586,40324;3068322
38185,1898207;1981510
77004,2775013;2775014
232597,831;657324
270884,1280;1282
303597,851;469604
317472,1849976;1849977;1849978;1849979;1849984;184999...


In [17]:
#for rows that have multiple taxid seperated by ; take the first taxid as represetnative
df_tax['taxid_rep'] = df_tax['taxid'].str.split(';').str[0]
mask = df_tax['taxid_rep'].str.contains(';', na=False) 
df_tax[mask]

/tmp/ipykernel_3948948/2354423992.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tax['taxid_rep'] = df_tax['taxid'].str.split(';').str[0]


,taxid,taxid_rep


In [18]:
df = pd.DataFrame(columns=['taxid_rep','phylum','class','order','family','genus', 'species'])

df_tax['taxid_rep'] = df_tax['taxid_rep'].astype(int)

for taxid in df_tax['taxid_rep']:
    try:
        taxa_dict = taxopy.Taxon(taxid, taxdb).rank_name_dictionary
        # Append only the selected items
        selected_data = {key: taxa_dict.get(key, 'N/A') for key in ['taxid_rep','phylum','class','order','family','genus', 'species']}
        selected_data['taxid_rep'] = taxid
        df = df._append(selected_data, ignore_index=True)
        
    except Exception as e:
        print(f"An error occurred for taxid '{taxid}': {e}")
        continue

/tmp/ipykernel_3948948/1308573555.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tax['taxid_rep'] = df_tax['taxid_rep'].astype(int)


In [19]:
#open nt_prok_blastn_out.txt and merge taxid lineages onto it

df_tax = pd.read_csv('nt_prok_blastn_out/nt_prok_blastn_out.txt', sep='\t')
df_tax[['taxname', 'taxid']] = df_tax['taxid'].str.split('_\$_', expand=True)
df_tax['taxid_rep'] = df_tax['taxid'].str.split(';').str[0]
df_tax['taxid_rep'] = df_tax['taxid_rep'].astype(int)

df_tax

,qseqid,taxid,file,taxname,taxid_rep
0,NODE_1001_length_5588_cov_8.509489,37923,250513Pat_D25-7849,"Rothia kristinae strain ND-2018N chromosome, c...",37923
1,NODE_1004_length_5581_cov_13.102787,1261,250513Pat_D25-7849,Peptostreptococcus anaerobius strain SB204 chr...,1261
2,NODE_1005_length_5575_cov_7.503623,37923,250513Pat_D25-7849,Rothia kristinae strain FDAARGOS_864 chromosom...,37923
3,NODE_1006_length_5564_cov_7.022690,1747,250513Pat_D25-7849,"Cutibacterium acnes SZ2 DNA, complete genome",1747
4,NODE_1007_length_5536_cov_23.626893,423477,250513Pat_D25-7849,Veillonella rogosae strain AC2811 AN NA 2 chro...,423477
...,...,...,...,...,...
318606,NODE_997_length_5892_cov_14.912798,2666183,250930Pat_D25-12478,Pseudomonas juntendi strain 18091276 chromosom...,2666183
318607,NODE_998_length_5889_cov_7.993315,577,250930Pat_D25-12478,Raoultella terrigena strain Res13-Abat-PEB01-P...,577
318608,NODE_999_length_5885_cov_15.724357,39778,250930Pat_D25-12478,Veillonella dispar strain NCTC11831 genome ass...,39778
318609,NODE_99_length_89346_cov_114.083771,1134687,250930Pat_D25-12478,Klebsiella michiganensis strain RHBSTW-00676 c...,1134687


In [20]:
dfm = pd.merge(df_tax, df, on = 'taxid_rep', how = 'left')
dfm

,qseqid,taxid,file,taxname,taxid_rep,phylum,class,order,family,genus,species
0,NODE_1001_length_5588_cov_8.509489,37923,250513Pat_D25-7849,"Rothia kristinae strain ND-2018N chromosome, c...",37923,Actinomycetota,Actinomycetes,Micrococcales,Micrococcaceae,Rothia,Rothia kristinae
1,NODE_1004_length_5581_cov_13.102787,1261,250513Pat_D25-7849,Peptostreptococcus anaerobius strain SB204 chr...,1261,Bacillota,Clostridia,Peptostreptococcales,Peptostreptococcaceae,Peptostreptococcus,Peptostreptococcus anaerobius
2,NODE_1005_length_5575_cov_7.503623,37923,250513Pat_D25-7849,Rothia kristinae strain FDAARGOS_864 chromosom...,37923,Actinomycetota,Actinomycetes,Micrococcales,Micrococcaceae,Rothia,Rothia kristinae
3,NODE_1006_length_5564_cov_7.022690,1747,250513Pat_D25-7849,"Cutibacterium acnes SZ2 DNA, complete genome",1747,Actinomycetota,Actinomycetes,Propionibacteriales,Propionibacteriaceae,Cutibacterium,Cutibacterium acnes
4,NODE_1007_length_5536_cov_23.626893,423477,250513Pat_D25-7849,Veillonella rogosae strain AC2811 AN NA 2 chro...,423477,Bacillota,Negativicutes,Veillonellales,Veillonellaceae,Veillonella,Veillonella rogosae
...,...,...,...,...,...,...,...,...,...,...,...
331984,NODE_997_length_5892_cov_14.912798,2666183,250930Pat_D25-12478,Pseudomonas juntendi strain 18091276 chromosom...,2666183,Pseudomonadota,Gammaproteobacteria,Pseudomonadales,Pseudomonadaceae,Pseudomonas,Pseudomonas juntendi
331985,NODE_998_length_5889_cov_7.993315,577,250930Pat_D25-12478,Raoultella terrigena strain Res13-Abat-PEB01-P...,577,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Raoultella,Raoultella terrigena
331986,NODE_999_length_5885_cov_15.724357,39778,250930Pat_D25-12478,Veillonella dispar strain NCTC11831 genome ass...,39778,Bacillota,Negativicutes,Veillonellales,Veillonellaceae,Veillonella,Veillonella dispar
331987,NODE_99_length_89346_cov_114.083771,1134687,250930Pat_D25-12478,Klebsiella michiganensis strain RHBSTW-00676 c...,1134687,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Klebsiella,Klebsiella michiganensis


In [21]:
dfm = dfm.drop_duplicates()
dfm

,qseqid,taxid,file,taxname,taxid_rep,phylum,class,order,family,genus,species
0,NODE_1001_length_5588_cov_8.509489,37923,250513Pat_D25-7849,"Rothia kristinae strain ND-2018N chromosome, c...",37923,Actinomycetota,Actinomycetes,Micrococcales,Micrococcaceae,Rothia,Rothia kristinae
1,NODE_1004_length_5581_cov_13.102787,1261,250513Pat_D25-7849,Peptostreptococcus anaerobius strain SB204 chr...,1261,Bacillota,Clostridia,Peptostreptococcales,Peptostreptococcaceae,Peptostreptococcus,Peptostreptococcus anaerobius
2,NODE_1005_length_5575_cov_7.503623,37923,250513Pat_D25-7849,Rothia kristinae strain FDAARGOS_864 chromosom...,37923,Actinomycetota,Actinomycetes,Micrococcales,Micrococcaceae,Rothia,Rothia kristinae
3,NODE_1006_length_5564_cov_7.022690,1747,250513Pat_D25-7849,"Cutibacterium acnes SZ2 DNA, complete genome",1747,Actinomycetota,Actinomycetes,Propionibacteriales,Propionibacteriaceae,Cutibacterium,Cutibacterium acnes
4,NODE_1007_length_5536_cov_23.626893,423477,250513Pat_D25-7849,Veillonella rogosae strain AC2811 AN NA 2 chro...,423477,Bacillota,Negativicutes,Veillonellales,Veillonellaceae,Veillonella,Veillonella rogosae
...,...,...,...,...,...,...,...,...,...,...,...
331984,NODE_997_length_5892_cov_14.912798,2666183,250930Pat_D25-12478,Pseudomonas juntendi strain 18091276 chromosom...,2666183,Pseudomonadota,Gammaproteobacteria,Pseudomonadales,Pseudomonadaceae,Pseudomonas,Pseudomonas juntendi
331985,NODE_998_length_5889_cov_7.993315,577,250930Pat_D25-12478,Raoultella terrigena strain Res13-Abat-PEB01-P...,577,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Raoultella,Raoultella terrigena
331986,NODE_999_length_5885_cov_15.724357,39778,250930Pat_D25-12478,Veillonella dispar strain NCTC11831 genome ass...,39778,Bacillota,Negativicutes,Veillonellales,Veillonellaceae,Veillonella,Veillonella dispar
331987,NODE_99_length_89346_cov_114.083771,1134687,250930Pat_D25-12478,Klebsiella michiganensis strain RHBSTW-00676 c...,1134687,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Klebsiella,Klebsiella michiganensis


In [22]:
dfm.to_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxids.txt", sep='\t', index = False)

In [23]:
# dfm['cov'] = dfm['qseqid'].str.extract(r'cov_(\d+)').astype(float)
# dfm
# filtered_dfm = dfm[dfm['cov'] >= 20]
# len(filtered_dfm)
# filtered_dfm.to_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxids.txt", sep='\t', index = False)

In [24]:
df_metagenome = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxids.txt", sep='\t')
df_metagenome['length'] = df_metagenome['qseqid'].str.extract(r'length_(\d+)_cov').astype(int)


pivot_metagenome = pd.pivot_table(
    df_metagenome,
    index='file',
    columns='species',
    values='length',
    aggfunc='sum',
    fill_value=0
)

pivot_metagenome.to_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_species_pivot.txt", sep='\t')

pivot_metagenome = pd.pivot_table(
    df_metagenome,
    index='file',
    columns='genus',
    values='length',
    aggfunc='sum',
    fill_value=0
)

pivot_metagenome.to_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_genus_pivot.txt", sep='\t')


In [23]:
df = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxid_species_pivot.txt", sep='\t', index_col='file').T
df = df.reset_index()
df.rename(columns={'index': 'species'}, inplace=True)
df.to_csv("nt_prok_blastn_out/species_mgx_otu.txt", sep = '\t', index = False)

df_tax = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxids.txt", sep='\t')
df_tax = df_tax[['phylum','class','order','family','genus','species']]
df_tax = df_tax.drop_duplicates(subset=['species'], keep='first')
df_tax['Species'] = df_tax['species']
column_to_move = 'Species'
col = df_tax.pop(column_to_move)
df_tax.insert(0, column_to_move, col)
df_tax.to_csv("nt_prok_blastn_out/species_mgx_tax.txt", sep = '\t', index = False)
